# 01x · 차원별 단순 랜덤 워크의 재귀성 — Pólya의 정리 (Recurrence of Simple Random Walk by Dimension — Pólya's Theorem)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Lawler 2e §2.2 (재귀 기준 $\sum p^n(0,0)$과 스털링으로 차원별 판정) · Norris §1.6 (Thm 1.6.1–1.6.3) · Ross 12e §4.3 (Prop. 4.1, 대칭 랜덤 워크 예제) · Durrett 3e §1.10 (무한 상태공간) · G&S 4e §5.3 (생성함수 $F=1-1/G$, 첫 귀환시간) |
| 선수 노트북 | 01b, 01g, 00a (Recall 문제는 01g·01f) |
| 예상 소요 | 90분 (선택) |
| 상태 | draft |

01b에서 유한 사슬의 닫힌 클래스는 자동으로 재귀였다. 상태공간이 무한하면 그 자동성이 사라진다: 닫혀 있고 기약인데도 **영영 돌아오지 않을 수 있다**. 가장 유명한 예가 $\mathbb Z^d$ 위의 단순 대칭 랜덤 워크다 — 1·2차원에서는 반드시 원점으로 돌아오지만 3차원부터는 34%만 돌아온다(Pólya, 1921). 이 노트북은 01b의 재귀 기준 $\sum_np^n(0,0)=\infty$를 무한 사슬에 그대로 적용해 이 사실을 **정확한 조합 공식 + 스털링 근사**로 증명하고, 첫 귀환시간 분포를 재귀식으로 정확히 계산해 시뮬레이션과 대조한다. 덤으로 유한 사슬에서는 볼 수 없던 현상 — **재귀인데 평균 귀환시간이 무한한** 영재귀 — 를 1차원에서 직접 목격한다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy.special import gammaln, logsumexp
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close

SEED, rng = rng_for("01x")
setup_plots()
N_PATHS = scale(20_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
N_STEPS = 1000            # 시뮬레이션 지평 (걸음 수)
N_MAX = 500               # 정확 계산: 이중 걸음 n = 1..500  (2n = 1000 걸음)
CHUNK = 2000              # 경로 청크 크기 (메모리, 함정 4)
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (01g) 소멸확률 $q$의 특징(pgf로)과 $m\le1$일 때의 값은?

<details><summary>정답</summary>

$q$는 $\varphi(s)=s$의 $[0,1]$ 안 **최소 근**이고, $q=\lim_n\varphi_n(0)$ (0에서 출발한 반복). $m=\varphi'(1)\le1$이면 $q=1$ (확률 1로 소멸).

</details>

**Q2.** (01g) $\varphi_{n+1}$과 $\varphi_n$의 관계, 그리고 $EZ_n$은?

<details><summary>정답</summary>

$\varphi_{n+1}=\varphi\circ\varphi_n$ (세대별 합성); $EZ_n=m^n$.

</details>

**Q3.** (01g) 임계 과정($m=1$)에서 평균은 1인데 왜 확률 1로 소멸하는가? 생존확률의 감쇠율은?

<details><summary>정답</summary>

소수의 폭발 경로가 평균을 지탱하고 대부분의 경로는 소멸한다(평균과 전형적 경로의 괴리). $P(Z_n>0)\sim 2/(\sigma^2 n)$ — 다항식 꼬리이므로 느리게 죽는다. 이 노트북의 1차원 걷기도 같은 그림이다: 확률 1로 돌아오지만 꼬리가 $n^{-1/2}$로 무겁다.

</details>

**Q4.** (01f) MCMC에서 $\lambda_2\approx1$이면 무엇이 문제인가?

<details><summary>정답</summary>

자기상관 시간 $\tau\le(1+\lambda_2)/(1-\lambda_2)$가 폭발해 유효 표본 수가 거의 0이 된다(쌍봉 목표에서 한 봉우리에 갇힘).

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **단순 대칭 랜덤 워크 (simple symmetric random walk, SRW) on $\mathbb Z^d$**: $S_n=\sum_{i\le n}X_i$, $X_i$ i.i.d. 균등 on $2d$개 단위벡터 $\{\pm e_1,\dots,\pm e_d\}$. 기약(irreducible)이고 주기 2인 **무한 상태** 마르코프 사슬이다. $p_n(0,0)=P(S_n=0)$은 홀수 $n$에서 0 (좌표합의 홀짝이 매 걸음 바뀐다).
- **귀환시간(return time)** $T=\min\{n\ge1:S_n=0\}$, **귀환확률** $\rho=P(T<\infty)$; 재귀 $\iff\rho=1$. **그린 함수(Green's function)** $G=\sum_{n\ge0}p_n(0,0)=E[\text{원점 방문 횟수(출발 포함)}]=1/(1-\rho)$ (01b 정리 1).
- **첫 귀환 재귀식(first-return decomposition)**: 시각 $n$에 원점에 있는 경로를 **첫** 귀환 시각 $k$로 분해하면
  $$p_n(0,0)=\sum_{k=1}^n f_k\,p_{n-k}(0,0),\qquad f_k=P(T=k),$$
  즉 $f_n=p_n-\sum_{k=1}^{n-1}f_kp_{n-k}$. 생성함수로 쓰면 $G(s)=1+F(s)G(s)$, 즉 $F(s)=1-1/G(s)$ (G&S 4e §5.3).
- **닫힌 식**: $d=1$: $p_{2n}(0,0)=\binom{2n}{n}2^{-2n}$ (오른쪽 $n$번, 왼쪽 $n$번). $d=2$: $p_{2n}(0,0)=\big(\binom{2n}n2^{-2n}\big)^2$ (45° 회전 $(x+y,\,x-y)$로 두 **독립** 1차원 걷기). $d=3$: $p_{2n}(0,0)=\binom{2n}n2^{-2n}\sum_{j+k+l=n}\Big(\dfrac{n!}{j!k!l!}3^{-n}\Big)^2$ (각 축에서 $\pm$ 걸음 수가 같아야 하고, 축 배정은 다항계수).
- **스털링(Stirling)**: $\binom{2n}n4^{-n}\sim(\pi n)^{-1/2}$. 따라서 $p_{2n}(0,0)\asymp n^{-d/2}$: $\sum_n n^{-d/2}$는 $d\le2$에서 발산, $d\ge3$에서 수렴.
- **영재귀(null recurrent)**: 재귀이지만 $ET=\infty$. **양재귀(positive recurrent)**: $ET<\infty$. 유한 기약 사슬은 항상 양재귀($E_xT_x=1/\pi(x)<\infty$, 01d); 무한 사슬에서만 영재귀가 가능하다.

### 2.2 정리 1 — 재귀 기준 (무한 상태공간) (Ross 12e §4.3 Prop. 4.1, Norris §1.6, Durrett 3e §1.10)

> 임의의 (시간 동질) 마르코프 사슬에서 상태 $x$에 대해
> $$\sum_{n\ge0}p^n(x,x)=\infty\iff x\text{ 재귀},\qquad \sum_{n\ge0}p^n(x,x)<\infty\iff x\text{ 일시},$$
> 그리고 일시일 때 $\rho_{xx}=1-1/\sum_np^n(x,x)$.

**가정이 왜 필요한가**
- 강마르코프 성질(귀환할 때마다 새로 시작): 방문 횟수 $V_x$가 기하분포가 되는 근거 — 01b와 동일.
- 상태공간의 **유한성은 불필요**: 그래서 무한 사슬에도 쓸 수 있다. 다만 유한 사슬에서는 '닫힌 클래스 ⇒ 재귀'가 자동이었지만(01b 정리 3), 무한에서는 이 급수 판정이 **실제로 필요**하다 — 이 노트북의 주제.

<details><summary>증명</summary>

01b 정리 1의 재진술이다. $V_x=\sum_{n\ge0}\mathbf 1\{X_n=x\}$ (출발 포함).

**(i)** $k$번째 귀환 시각을 $T^{(1)}=T_x$, $T^{(k+1)}=\min\{n>T^{(k)}:X_n=x\}$로 두면 강마르코프 성질에 의해 $T^{(k)}$ 이후의 사슬은 $x$에서 새로 출발한 사슬과 같은 법칙을 따르므로 $P_x(T^{(k+1)}<\infty)=P_x(T^{(k)}<\infty)\,\rho_{xx}$, 귀납적으로 $P_x(T^{(k)}<\infty)=\rho_{xx}^k$.

**(ii)** $V_x\ge k+1\iff T^{(k)}<\infty$이므로 $P_x(V_x\ge k+1)=\rho_{xx}^k$, 꼬리합 공식으로
$$E_xV_x=\sum_{k\ge0}P_x(V_x>k)=\sum_{k\ge0}\rho_{xx}^k=\begin{cases}1/(1-\rho_{xx})&\rho_{xx}<1\\ \infty&\rho_{xx}=1.\end{cases}$$

**(iii)** 한편 각 항이 비음이므로 단조수렴(Tonelli)으로 $E_xV_x=\sum_{n\ge0}P_x(X_n=x)=\sum_{n\ge0}p^n(x,x)$.

(ii)와 (iii)을 합치면 $\rho_{xx}=1\iff\sum_np^n(x,x)=\infty$이고, $\rho_{xx}<1$이면 $\sum_np^n(x,x)=1/(1-\rho_{xx})$, 즉 $\rho_{xx}=1-1/\sum_np^n(x,x)$. 어디에도 상태공간이 유한하다는 가정은 쓰이지 않았다. $\square$

</details>

### 2.3 정리 2 — Pólya의 정리 (Norris §1.6 Thm 1.6.1–1.6.3, Lawler 2e §2.2)

> $\mathbb Z^d$ 위의 SRW는 $d=1,2$에서 재귀, $d\ge3$에서 일시다. $d=3$의 귀환확률은 $\rho_3=1-1/G_3\approx0.3405$ ($G_3=\sum_np_{2n}(0,0)\approx1.5164$, Watson 상수).

**가정이 왜 필요한가**
- 대칭·단순($\pm$단위벡터, 균등): $p_{2n}(0,0)$의 정확한 조합 공식과 스털링 점근이 성립하는 근거. 대칭을 깨면(편향 걷기, E3) **$d=1$에서도 일시**: $\rho=1-|p-q|$.
- $d\ge3$에서 $\sum_nn^{-d/2}<\infty$: 3차원부터 급수가 수렴한다는 순수 해석적 사실. 정리 1이 이 해석적 사실을 확률적 결론(일시)으로 번역한다.

**증명 스케치.**
- $d=1$: $p_{2n}(0,0)=\binom{2n}n4^{-n}\sim(\pi n)^{-1/2}$ (스털링 $n!\sim\sqrt{2\pi n}(n/e)^n$), $\sum_nn^{-1/2}=\infty$ ⇒ 재귀.
- $d=2$: 좌표 회전 $(U,V)=(X+Y,\,X-Y)$를 쓰면 $U,V$는 각각 독립인 1차원 SRW다 (네 방향 걸음 $(\pm1,0),(0,\pm1)$이 $(\pm1,\pm1)$의 네 조합으로 옮겨진다). $S_{2n}=0\iff U_{2n}=V_{2n}=0$이므로 $p_{2n}=\big(\binom{2n}n4^{-n}\big)^2\sim(\pi n)^{-1}$ — 조화급수, **아슬아슬하게** 발산 ⇒ 재귀.
- $d=3$: $\sum_{j+k+l=n}\frac{n!}{j!k!l!}3^{-n}=1$ (다항정리)이고 최댓값 항은 $j,k,l\approx n/3$에서 $\asymp n^{-1}$이므로 제곱합 $\le\max\times\sum=\max\asymp n^{-1}$. 따라서 $p_{2n}\lesssim n^{-1/2}\cdot n^{-1}=n^{-3/2}$, $\sum n^{-3/2}<\infty$ ⇒ 일시. 정확한 점근은 $p_{2n}\sim2\,(3/(4\pi n))^{3/2}$ (계수 $c_3=2(3/4\pi)^{3/2}\approx0.2333$; 국소 중심극한정리).
- $d>3$: 처음 세 좌표만 보면(그 좌표들이 움직이는 시각만 세면) 3차원 SRW가 되므로 원점 귀환은 더 어렵다 — 3차원 투영 논법.

### 2.4 정리 3 — 1차원 귀환시간 꼬리 (G&S 4e §5.3)

> $d=1$ SRW: $P(T>2n)=P(S_{2n}=0)=\binom{2n}n4^{-n}\sim(\pi n)^{-1/2}$, $\quad P(T=2n)=\dfrac{\binom{2n}n}{4^n(2n-1)}\sim\dfrac1{2\sqrt\pi\,n^{3/2}}$; 따라서 $ET=\sum_n2n\,f_{2n}=\infty$ (영재귀).

**가정이 왜 필요한가**
- 대칭 $p=1/2$: 반사 원리·투표 정리가 성립하고 $G(s)=(1-s^2)^{-1/2}$가 되는 조건. 편향이면 $G(1)<\infty$.
- $\sum_n n\cdot n^{-3/2}=\sum_nn^{-1/2}$의 발산이 $ET=\infty$의 이유. 꼬리 $P(T>2n)\sim n^{-1/2}$는 '한 번도 안 돌아온 확률'이 다항식으로 느리게 죽는다는 뜻이다.

**증명 스케치.** $\binom{2n}n4^{-n}=(-1)^n\binom{-1/2}{n}$이므로 $G(s)=\sum_np_{2n}s^{2n}=(1-s^2)^{-1/2}$. 그러면 $F(s)=1-1/G(s)=1-\sqrt{1-s^2}$이고 이항급수로 전개하면 $f_{2n}=\binom{2n}n/(4^n(2n-1))$. $F(1)=1$ ⇒ 재귀; $F'(1)=\lim_{s\to1}s/\sqrt{1-s^2}=\infty$ ⇒ $ET=\infty$. 꼬리 항등식 $P(T>2n)=p_{2n}(0,0)$은 $F=1-1/G$의 계수 비교($p_{2n}=\sum_kf_{2k}p_{2n-2k}$)와 귀납으로, 또는 투표 정리로 얻는다(E2). 노트북에서는 재귀식으로 $f$를 계산해 닫힌 식과 `allclose`로 확인한다.

### 2.5 직관

재귀성은 '원점 방문 횟수의 기댓값이 무한한가'와 같고, 그 기댓값은 $\sum_np_n(0,0)$이다. $n$걸음 뒤 걷기는 반지름 $\sqrt n$의 공 안에 퍼져 있으므로(중심극한정리) 그 공 안의 $\asymp n^{d/2}$개 격자점 중 특정 한 점(원점)에 있을 확률은 $\asymp n^{-d/2}$: 1차원은 $n^{-1/2}$(합 발산), 2차원은 $n^{-1}$(조화급수, 아슬아슬하게 발산), 3차원은 $n^{-3/2}$(수렴). "취한 사람은 집에 돌아오지만 취한 새는 돌아오지 못한다"(Kakutani). 3차원 새는 34%만 돌아온다.

1차원은 돌아오긴 하지만 평균 귀환시간이 무한하다 — 유한 사슬에서는 볼 수 없던 영재귀. 1000걸음 안에 못 돌아올 확률은 2.5%뿐인데, 그 2.5%가 무한한 기댓값을 만든다(꼬리 $n^{-1/2}$는 적분하면 발산).

### 2.6 함정

1. 귀환은 짝수 시각에만 가능하므로 $\sum_np_n(0,0)$은 $n=2k$만 더한다. 홀수까지 세면 0을 더할 뿐이지만, 근사 $(\pi n)^{-1/2}$의 $n$은 **걸음 수의 절반**(이중 걸음 수)임에 주의. 이 노트북의 배열 인덱스는 모두 이중 걸음 $n$이다: `p1[k]` $=p_{2k}(0,0)$.
2. 시뮬레이션의 '1000걸음 안 귀환 비율'은 $\rho$가 아니라 $P(T\le1000)$이다. $d=1$: $0.9748$ (vs $\rho=1$), $d=3$: $0.3315$ (vs $\rho_3=0.3405$). 꼬리가 $n^{-1/2}$로 무거워 지평을 늘려도 느리게 붙는다.
3. 3차원 $p_{2n}(0,0)$의 합 공식을 그대로 계산하면 $n\approx500$에서 오버플로($500!\approx10^{1134}$, 다항계수의 제곱 $\sim9^n$; float64 최대는 $\approx1.8\times10^{308}$). 로그 공간(`gammaln` + `logsumexp`)에서 계산할 것.
4. 메모리: $20000$경로$\times1000$걸음$\times3$좌표를 int64로 한 번에 만들면 480 MB. 경로를 청크(2000개)로 나눠 처리한다.
5. '재귀'와 '유한 평균 귀환시간'은 다르다. 1·2차원 SRW는 재귀이지만 영재귀 — 정상분포가 없다($\pi(x)=1/E_xT_x=0$; 01d의 존재 정리에 '유한'이 붙은 이유).

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요 — (1) $\sum p^n(0,0)=E[\text{방문 횟수}]$인 이유, (2) 차원에 따라 $p_{2n}(0,0)\asymp n^{-d/2}$인 직관, (3) '재귀인데 $ET=\infty$'가 어떻게 가능한지를 각각 두 문장으로)

## 3. Predict — 코드를 돌리기 전에 예측

코드를 돌리기 전에 숫자를 적어 보세요. 틀려도 좋습니다 — 어디서 얼마나 틀렸는지가 5절의 학습 재료입니다.

In [ ]:
predictions = {
    # 1차원 SRW가 20걸음 뒤 원점에 있을 확률 p_20(0,0)은? (힌트: Ehrenfest(20)의 pi(10)과 같은 수)
    "p20_d1": None,
    # 1차원 SRW가 1000걸음 안에 원점으로 돌아올 확률 P(T <= 1000)은? (= 1 - P(S_1000 = 0))
    "p_return_by_1000_d1": None,
    # 2차원에서 P(T <= 1000)은? (궁극적으로는 1이지만 매우 느리게 붙는다)
    "p_return_by_1000_d2": None,
    # 3차원에서 P(T <= 1000)은?
    "p_return_by_1000_d3": None,
    # 3차원의 궁극 귀환확률 rho_3 = 1 - 1/G_3 은?
    "rho_3": None,
    # 1차원에서 100걸음 뒤에도 아직 한 번도 안 돌아왔을 확률 P(T > 100)은? (근사 (pi*50)^(-1/2)와 비교)
    "p_T_gt_100_d1": None,
    # 3차원에서 1..1000걸음 동안 원점 방문 횟수의 기댓값은? (궁극값은 G_3 - 1 = 0.5164)
    "visits_1000_d3": None,
}

## 4. Simulate — 시뮬레이션

### 4.1 경로 그림 — 취한 사람과 취한 새

측정에 앞서 걷기가 어떻게 생겼는지 본다. 왼쪽: 1차원 경로 5개(1000걸음). 오른쪽: 2차원 경로 하나의 자취. 1차원 경로가 0을 몇 번이나 가로지르는지, 2차원 자취가 출발점 근처를 얼마나 성기게 다시 밟는지 보라.

In [ ]:
steps1 = rng.choice([-1, 1], size=(5, N_STEPS))
paths1 = np.concatenate([np.zeros((5, 1), dtype=int), steps1.cumsum(1)], axis=1)
k2 = rng.integers(0, 4, size=N_STEPS)                       # 0,1: x축 / 2,3: y축
steps2 = np.zeros((N_STEPS, 2), dtype=int)
steps2[np.arange(N_STEPS), k2 // 2] = 2 * (k2 % 2) - 1
path2 = np.vstack([[0, 0], steps2.cumsum(0)])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for i in range(5):
    ax1.plot(paths1[i], lw=0.8)
ax1.axhline(0, color="k", lw=0.8)
ax1.set(title="d = 1: five SRW paths, 1000 steps", xlabel="step n", ylabel="S_n")
ax2.plot(path2[:, 0], path2[:, 1], lw=0.6, alpha=0.8)
ax2.plot(0, 0, "ro", ms=6, label="origin")
ax2.plot(*path2[-1], "ks", ms=5, label="S_1000")
ax2.set(title="d = 2: one SRW path, 1000 steps", xlabel="x", ylabel="y")
ax2.set_aspect("equal"); ax2.legend()
plt.tight_layout(); plt.show()
print(f"d=1 paths: visits to 0 in steps 1..{N_STEPS} =", (paths1[:, 1:] == 0).sum(1))

### 4.2 정확 계산 — $p_{2n}(0,0)$, 첫 귀환 pmf, 그린 함수

닫힌 식을 **로그 공간**에서 계산한다(함정 3). $d=3$의 다항계수 제곱합은 $\sum_{j+k+l=n}\big(\frac{n!}{j!k!l!}\big)^2=\sum_j\binom nj^2\binom{2(n-j)}{n-j}$로 한 겹 줄여 계산한다 ($\frac{n!}{j!k!l!}=\binom nj\binom{n-j}k$이고 $\sum_k\binom mk^2=\binom{2m}m$). 첫 귀환 pmf는 재귀식 $f_n=p_n-\sum_{k<n}f_kp_{n-k}$ (인덱스 = 이중 걸음 수).

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def logp2n_d1(n):
    """log of C(2n, n) / 4^n = log p_{2n}(0,0) for d = 1 (vectorized over n)."""
    n = np.asarray(n, dtype=float)
    return gammaln(2 * n + 1) - 2 * gammaln(n + 1) - 2 * n * np.log(2)

def p2n_d3(n):
    """p_{2n}(0,0) for d = 3: C(2n,n)4^-n * 9^-n * sum_j C(n,j)^2 C(2(n-j), n-j), in log space."""
    if n == 0:
        return 1.0
    j = np.arange(n + 1); m = n - j
    logterm = 2 * (gammaln(n + 1) - gammaln(j + 1) - gammaln(m + 1)) + gammaln(2 * m + 1) - 2 * gammaln(m + 1)
    return float(np.exp(logp2n_d1(n) - 2 * n * np.log(3) + logsumexp(logterm)))

def first_return_pmf(p):
    """f[k] = P(T = 2k) from p[k] = p_{2k}(0,0) via f_k = p_k - sum_{j<k} f_j p_{k-j}; f[0] = 0."""
    f = np.zeros_like(p)
    for k in range(1, len(p)):
        f[k] = p[k] - np.dot(f[1:k], p[k - 1:0:-1])
    return f

def loglog_slope(x, y, lo, hi):
    """Least-squares slope of log y vs log x over index range lo..hi (inclusive)."""
    return float(np.polyfit(np.log(x[lo:hi + 1]), np.log(y[lo:hi + 1]), 1)[0])

In [ ]:
nn = np.arange(N_MAX + 1)                       # 이중 걸음 수 n (2n 걸음)
p1 = np.exp(logp2n_d1(nn))                      # d = 1
p2 = p1 ** 2                                    # d = 2
p3 = np.array([p2n_d3(n) for n in nn])          # d = 3
f1, f2, f3 = first_return_pmf(p1), first_return_pmf(p2), first_return_pmf(p3)

# 검산: d=1 재귀식 == 닫힌 식 f_2n = C(2n,n) / (4^n (2n-1)); d=3 작은 n 손계산
f1_closed = np.exp(logp2n_d1(nn[1:]) - np.log(2 * nn[1:] - 1))
assert np.allclose(f1[1:], f1_closed, rtol=1e-8)
assert np.isclose(f1.sum(), 1 - p1[N_MAX])                      # P(T <= 2N) = 1 - p_{2N}  (정리 3)
assert np.isclose(p3[1], 1 / 6) and np.isclose(p3[2], 5 / 72)

# 그린 함수 G_d = sum_n p_{2n}: 부분합 + d=3 꼬리 보정 (sum_{n>N} c n^{-3/2} ~ 2c / sqrt(N + 1/2))
c3 = 2 * (3 / (4 * np.pi)) ** 1.5
G3_partial = p3.sum()
tail3 = 2 * c3 / np.sqrt(N_MAX + 0.5)
G3 = G3_partial + tail3
rho3_exact = 1 - 1 / G3
assert abs(G3 - 1.516386) < 1e-4                                # Watson 상수 G_3 = 1.516386...
print(f"partial sums G_d(500): d=1 {p1.sum():.3f}, d=2 {p2.sum():.4f}, d=3 {G3_partial:.6f}")
print(f"G_3 = {G3_partial:.6f} + tail {tail3:.6f} = {G3:.6f}  (Watson 1.516386) -> rho_3 = {rho3_exact:.6f}")
print(f"f_2, f_4, f_6 (d=1) = {f1[1]:.4f}, {f1[2]:.4f}, {f1[3]:.4f}   (1/2, 1/8, 1/16)")
print(f"P(T <= 1000): d=1 {f1.sum():.6f}, d=2 {f2.sum():.6f}, d=3 {f3.sum():.6f}")

$d=1,2$의 부분합은 $N$과 함께 계속 커지고($\sqrt N$, $\log N$ 꼴), $d=3$만 상수 $G_3$에 수렴한다. 아래 로그-로그 그림에서 기울기 $-1/2,-1,-3/2$가 그 이유다.

In [ ]:
n_plot = nn[1:]
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.loglog(n_plot, p1[1:], "o", ms=2.5, label="d = 1 exact")
ax.loglog(n_plot, p2[1:], "s", ms=2.5, label="d = 2 exact")
ax.loglog(n_plot, p3[1:], "^", ms=2.5, label="d = 3 exact")
ax.loglog(n_plot, (np.pi * n_plot) ** -0.5, "k--", lw=1, label=r"$(\pi n)^{-1/2}$")
ax.loglog(n_plot, (np.pi * n_plot) ** -1.0, "k-.", lw=1, label=r"$(\pi n)^{-1}$")
ax.loglog(n_plot, c3 * n_plot ** -1.5, "k:", lw=1, label=r"$2(3/(4\pi n))^{3/2}$")
ax.set(xlabel="n  ($p_{2n}$: 2n steps)", ylabel="$p_{2n}(0,0)$",
       title="Return probabilities decay like n^(-d/2)")
ax.legend(ncol=2); plt.show()

### 4.3 시뮬레이션 — 첫 귀환시각과 방문 횟수

$d$차원 걷기를 청크 단위로 벡터화한다(함정 4). 각 걸음은 축 `k // 2`와 부호 `2*(k%2)-1`로 정하고, 시간 루프에서 위치를 갱신하며 원점 여부를 검사해 첫 귀환시각 $T$(없으면 `inf`)와 걸음 $1..N$ 동안의 방문 횟수를 기록한다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def first_return_and_visits(d, n_paths, n_steps, rng, chunk=2000):
    """d-dim SRW from the origin. Returns (T: first-return step, np.inf if none; visits: # of returns in steps 1..n_steps)."""
    T = np.full(n_paths, np.inf)
    visits = np.zeros(n_paths, dtype=int)
    for s in range(0, n_paths, chunk):
        m = min(chunk, n_paths - s)
        k = rng.integers(0, 2 * d, size=(m, n_steps))
        axis, sign = k // 2, 2 * (k % 2) - 1
        pos = np.zeros((m, d), dtype=np.int64)
        rows = np.arange(m)
        Tc, vc = np.full(m, np.inf), np.zeros(m, dtype=int)
        for t in range(n_steps):
            pos[rows, axis[:, t]] += sign[:, t]
            home = ~pos.any(axis=1)                 # 모든 좌표가 0
            vc += home
            Tc[home & np.isinf(Tc)] = t + 1
        T[s:s + m], visits[s:s + m] = Tc, vc
    return T, visits

In [ ]:
T1, visits1 = first_return_and_visits(1, N_PATHS, N_STEPS, rng, chunk=CHUNK)
T2, visits2 = first_return_and_visits(2, N_PATHS, N_STEPS, rng, chunk=CHUNK)
T3, visits3 = first_return_and_visits(3, N_PATHS, N_STEPS, rng, chunk=CHUNK)

est_ret1 = mc_proportion(np.isfinite(T1))
est_ret2 = mc_proportion(np.isfinite(T2))
est_ret3 = mc_proportion(np.isfinite(T3))
est_tail100 = mc_proportion(T1 > 100)
est_vis1 = mc_estimate(visits1)
est_vis3 = mc_estimate(visits3)
print(f"P(T <= {N_STEPS}):  d=1 {est_ret1}   d=2 {est_ret2}   d=3 {est_ret3}")
print(f"P(T > 100), d=1: {est_tail100}")
print(f"visits in steps 1..{N_STEPS}:  d=1 {est_vis1}   d=3 {est_vis3}")

$P(T\le n)$의 경험 누적분포(계단)와 정확한 $\sum_{k\le n/2}f_{2k}$(선)를 겹친다. $d=1,2$는 1로 기어 올라가고(2차원은 매우 느리게), $d=3$은 $\rho_3$ 아래에서 멈춘다.

In [ ]:
n_grid = np.arange(2, N_STEPS + 1, 2)
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for d, T, f, col in ((1, T1, f1, "C0"), (2, T2, f2, "C1"), (3, T3, f3, "C2")):
    Ts = np.sort(T[np.isfinite(T)])
    ecdf = np.searchsorted(Ts, n_grid, side="right") / T.size
    ax.step(n_grid, ecdf, where="post", color=col, lw=1.2, label=f"d = {d} MC (n={T.size})")
    ax.plot(n_grid, np.cumsum(f[1:]), color=col, ls="--", lw=1, label=f"d = {d} exact")
ax.axhline(rho3_exact, color="C2", ls=":", lw=1)
ax.text(2.2, rho3_exact + 0.015, f"rho_3 = {rho3_exact:.4f}", color="C2", fontsize=9)
ax.set(xscale="log", xlabel="n (steps)", ylabel="P(T <= n)", ylim=(0, 1.02),
       title="Return by step n: recurrent in d = 1, 2; transient in d = 3")
ax.legend(ncol=2, fontsize=8, loc="lower right"); plt.show()

### 4.4 1차원 꼬리 — 영재귀의 징후

생존함수 $P(T>2n)$(시뮬레이션과 정확값 $p_{2n}(0,0)$)과 첫 귀환 pmf $f_{2n}$을 로그-로그로 그린다. 기울기 $-1/2$와 $-3/2$: pmf가 $n^{-3/2}$이므로 $\sum_n 2n\,f_{2n}\asymp\sum n^{-1/2}=\infty$.

In [ ]:
n_lo, n_hi = 20, N_MAX
n_tail = np.arange(5, N_MAX + 1)
surv_mc = np.array([(T1 > 2 * n).mean() for n in n_tail])
slope_surv = loglog_slope(nn, p1, n_lo, n_hi)
slope_pmf = loglog_slope(nn, f1, n_lo, n_hi)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
ax1.loglog(n_tail, surv_mc, ".", ms=3, alpha=0.6, label="MC  P(T > 2n)")
ax1.loglog(n_tail, p1[n_tail], "-", lw=1.2, label="exact  $p_{2n}(0,0)$")
ax1.loglog(n_tail, (np.pi * n_tail) ** -0.5, "k--", lw=1, label=r"$(\pi n)^{-1/2}$")
ax1.set(xlabel="n", ylabel="$P(T > 2n)$", title=f"survival: exact-curve slope {slope_surv:.3f}"); ax1.legend()
ax2.loglog(n_tail, f1[n_tail], "-", lw=1.2, label="exact  $f_{2n} = P(T = 2n)$")
ax2.loglog(n_tail, 1 / (2 * np.sqrt(np.pi) * n_tail ** 1.5), "k--", lw=1, label=r"$1/(2\sqrt{\pi}\,n^{3/2})$")
ax2.set(xlabel="n", ylabel="$f_{2n}$", title=f"first-return pmf: exact-curve slope {slope_pmf:.3f}"); ax2.legend()
fig.suptitle("d = 1: null recurrence - heavy tail n^(-3/2), E[T] = infinity", y=1.02)
plt.tight_layout(); plt.show()
assert abs(slope_surv + 0.5) < 0.03 and abs(slope_pmf + 1.5) < 0.03

In [ ]:
print(f"fraction returned within {N_STEPS} steps (d=1): {np.isfinite(T1).mean():.4f}  (rho = 1, but P(T > 1000) = {p1[N_MAX]:.4f})")
for h in (10, 100, N_STEPS):
    print(f"  mean of T given T <= {h:5d}: {T1[T1 <= h].mean():8.2f}   (exact {(2 * nn[:h // 2 + 1] * f1[:h // 2 + 1]).sum() / f1[:h // 2 + 1].sum():8.2f})")
print("-> the conditional mean keeps growing with the horizon (~ sqrt(h)): E[T] = infinity (null recurrent)")

### 4.5 스털링 검증

In [ ]:
print("   n   C(2n,n)4^-n*sqrt(pi n)   p_2n^(2)*pi n   p_2n^(3)*n^1.5  (-> 1, 1, c_3=%.4f)" % c3)
for m in (10, 50, 100, 500):
    print(f"{m:4d}   {p1[m] * np.sqrt(np.pi * m):20.4f}   {p2[m] * np.pi * m:13.4f}   {p3[m] * m ** 1.5:14.4f}")

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식은 전부 4.2에서 코드로 계산한 값이다: $P(T\le1000)=\sum_{k\le500}f_{2k}$ (재귀식; $d=1$은 $1-p_{1000}$과 같다), $P(T>100)=p_{100}(0,0)$, 방문 횟수 기댓값 $=\sum_{n=1}^{500}p_{2n}(0,0)$.

In [ ]:
p_ret1_exact = float(1 - p1[N_MAX])          # = f1.sum()  (정리 3)
p_ret2_exact = float(f2.sum())
p_ret3_exact = float(f3.sum())
p_tail100_exact = float(p1[50])              # P(T > 100) = P(S_100 = 0)
vis1_exact = float(p1[1:].sum())
vis3_exact = float(p3[1:].sum())

rows = [
    {"name": "p_20(0,0), d=1 (not MC)", "predicted": predictions["p20_d1"], "estimate": float(p1[10]), "exact": float(p1[10])},
    {"name": "P(T <= 1000), d=1", "predicted": predictions["p_return_by_1000_d1"], "estimate": est_ret1, "exact": p_ret1_exact},
    {"name": "P(T <= 1000), d=2", "predicted": predictions["p_return_by_1000_d2"], "estimate": est_ret2, "exact": p_ret2_exact},
    {"name": "P(T <= 1000), d=3", "predicted": predictions["p_return_by_1000_d3"], "estimate": est_ret3, "exact": p_ret3_exact},
    {"name": "rho_3 = 1 - 1/G_3 (not MC)", "predicted": predictions["rho_3"], "estimate": rho3_exact, "exact": rho3_exact},
    {"name": "P(T > 100), d=1", "predicted": predictions["p_T_gt_100_d1"], "estimate": est_tail100, "exact": p_tail100_exact},
    {"name": "E[visits in 1..1000], d=1", "predicted": None, "estimate": est_vis1, "exact": vis1_exact},
    {"name": "E[visits in 1..1000], d=3", "predicted": predictions["visits_1000_d3"], "estimate": est_vis3, "exact": vis3_exact},
]
print(f"exact: P(T<=1000) = {p_ret1_exact:.6f} / {p_ret2_exact:.6f} / {p_ret3_exact:.6f} (d=1,2,3),  "
      f"P(T>100) = {p_tail100_exact:.6f} (approx {(np.pi * 50) ** -0.5:.6f}),  visits = {vis1_exact:.4f} / {vis3_exact:.6f}")
Markdown(compare_table(rows))

행별로 '불일치'가 뜻하는 것:
- **p_20(0,0)**: MC가 아니다(SE·CI 없음). 예측이 틀렸다면 $\binom{20}{10}/2^{20}$과 Ehrenfest $\pi(10)=\binom{20}{10}/2^{20}$이 같은 수임을 잊은 것.
- **P(T ≤ 1000), d=1**: 어긋나면 $T$를 걸음 $0$부터 세었거나(출발을 귀환으로 오인) 재귀식 인덱스(이중 걸음)가 밀린 것.
- **P(T ≤ 1000), d=2**: 어긋나면 45° 회전 논법($p^{(2)}_{2n}=(p^{(1)}_{2n})^2$)이 틀렸다는 뜻 — 4방향 걸음이 두 독립 1차원 걷기로 쪼개지는지 다시 보라.
- **P(T ≤ 1000), d=3**: 어긋나면 다항계수 제곱합(로그 공간)이 틀린 것; 작은 $n$ 검산($1/6$, $5/72$)이 통과했다면 축 배정 `k // 2`를 의심.
- **rho_3**: MC가 아니다(SE·CI 없음). 예측이 $0.33$ 근처였다면 $P(T\le1000)$과 $\rho$를 혼동한 것(함정 2).
- **P(T > 100), d=1**: 어긋나면 정리 3의 꼬리 항등식 $P(T>2n)=p_{2n}(0,0)$이 의심된다 — 대칭 $p=1/2$가 깨졌는지 확인.
- **E[visits], d=1 / d=3**: 어긋나면 정리 1의 $E[V]=\sum p_n(0,0)$ 자체가 의심된다. $d=1$은 분산이 커서(sd $\approx18$) SE도 크다.

In [ ]:
assert_close(est_ret1, p_ret1_exact, k=4, name="p_return_by_1000_d1")
assert_close(est_ret2, p_ret2_exact, k=4, name="p_return_by_1000_d2")
assert_close(est_ret3, p_ret3_exact, k=4, name="p_return_by_1000_d3")
assert_close(est_tail100, p_tail100_exact, k=4, name="p_T_gt_100_d1")
assert_close(est_vis1, vis1_exact, k=4, name="visits_1000_d1")
assert_close(est_vis3, vis3_exact, k=4, name="visits_1000_d3")
print("all checks passed")

## 6. 연습문제

### E1 계산

$p_{10}(0,0)$을 $d=1,2,3$에서 손으로(또는 계산기로) 구하시오. $d=3$은 $n=5$이므로 $\sum_{j+k+l=5}\big(\frac{5!}{j!k!l!}\big)^2$을 직접 더한다. 세 값의 비가 대략 $1:n^{-1/2}:n^{-1}$ 꼴임을 확인하시오. 다 푼 뒤 `p1[5], p2[5], p3[5]`로 검산하시오.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

- $d=1$: $\binom{10}5/2^{10}=252/1024=0.246094$.
- $d=2$: $(252/1024)^2=0.060562$.
- $d=3$: $\binom{10}5/2^{10}\cdot3^{-10}\sum(5!/(j!k!l!))^2$. 합은 $(j,k,l)$의 유형별로: $(5,0,0)$형 3개 $\times1^2$, $(4,1,0)$형 6개 $\times5^2$, $(3,2,0)$형 6개 $\times10^2$, $(3,1,1)$형 3개 $\times20^2$, $(2,2,1)$형 3개 $\times30^2$ $=3+150+600+1200+2700=4653$. 따라서 $0.246094\cdot4653/59049=0.019392$.

비: $0.2461:0.0606:0.0194\approx1:0.246:0.079$. $n=5$에서 $n^{-1/2}=0.447$, $n^{-1}=0.2$이니 거칠게 그 꼴이다 — 상수 $(\pi)^{-1/2}$ 등이 붙어 있고 $n=5$는 작다. 4.5의 스털링 표가 $n$이 커질수록 상수가 $1,1,c_3$으로 안정되는 것을 보여 준다.

검산 코드:
```python
from math import comb
s3 = 3 * 1 + 6 * 25 + 6 * 100 + 3 * 400 + 3 * 900          # 4653
print(comb(10, 5) / 2**10, (comb(10, 5) / 2**10) ** 2, comb(10, 5) / 2**10 * s3 / 3**10)
print(p1[5], p2[5], p3[5])       # 0.24609375 0.0605621337890625 0.019391932441700965
```

</details>

### E2 유도 후 시뮬레이션 검증

$d=1$에서 $G(s)=\sum_np_{2n}s^{2n}=(1-s^2)^{-1/2}$와 $F(s)=1-1/G(s)$로부터 $f_{2n}=\binom{2n}n/(4^n(2n-1))$과 $P(T>2n)=p_{2n}(0,0)$을 유도하시오. 그런 다음 4.3의 `T1`(`N_PATHS`개 경로)로 $P(T>100)$, $f_2=1/2$, $f_4=1/8$, $f_6=1/16$을 추정해 4 SE 안에서 일치하는지 `assert_close`로 확인하시오.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

**유도.** 일반화 이항계수로 $\binom{-1/2}{n}=(-1)^n\binom{2n}n4^{-n}$이므로 $G(s)=\sum_n\binom{2n}n4^{-n}s^{2n}=\sum_n\binom{-1/2}n(-s^2)^n=(1-s^2)^{-1/2}$. 따라서
$$F(s)=1-\frac1{G(s)}=1-\sqrt{1-s^2}=-\sum_{n\ge1}\binom{1/2}{n}(-s^2)^n,$$
그리고 $-\binom{1/2}n(-1)^n=\dfrac{\binom{2n}n}{(2n-1)4^n}$이므로 $f_{2n}=\binom{2n}n/(4^n(2n-1))$. $F(1)=1$ ⇒ 재귀. $F'(s)=s/\sqrt{1-s^2}\to\infty$ ⇒ $ET=\infty$.

**꼬리 항등식.** $F=1-1/G\iff FG=G-1$, 계수 비교: $p_{2n}=\sum_{k=1}^nf_{2k}\,p_{2n-2k}$ ($n\ge1$). $P(T>2n)=1-\sum_{k\le n}f_{2k}$이므로 보일 것은 $\sum_{k\le n}f_{2k}=1-p_{2n}$. 직접 계산이 가장 빠르다: $f_{2k}=\frac{\binom{2k}k}{4^k(2k-1)}=p_{2k-2}-p_{2k}$ (확인: $p_{2k}/p_{2k-2}=\frac{2k-1}{2k}$이므로 $p_{2k-2}-p_{2k}=p_{2k}\cdot\frac{1}{2k-1}=f_{2k}$). 망원합으로 $\sum_{k\le n}f_{2k}=p_0-p_{2n}=1-p_{2n}$. $\square$ (투표 정리를 쓰는 조합적 증명은 G&S 4e §5.3.)

시뮬레이션 코드 (SE: $P(T>100)$ 약 $0.0019$, $f_2$ 약 $0.0035$ at $n=20000$):
```python
targets = {"P(T>100)": (T1 > 100, p1[50]), "f_2": (T1 == 2, f1[1]),
           "f_4": (T1 == 4, f1[2]), "f_6": (T1 == 6, f1[3])}
for name, (event, exact) in targets.items():
    est = mc_proportion(event)
    print(f"{name:9s} {est}   exact {exact:.6f}")
    assert_close(est, exact, k=4, name=name)
# exact: 0.079589, 0.5, 0.125, 0.0625
```

</details>

### E3 가정을 깨보기

대칭성을 깨자: $d=1$에서 오른쪽 확률 $p=0.55$($q=0.45$)인 편향 걷기(biased walk). 귀환확률 $\rho=1-|p-q|$를 유도하고(힌트: $p_{2n}(0,0)=\binom{2n}n(pq)^n$, $G(1)=(1-4pq)^{-1/2}=1/|p-q|$), 1000걸음 시뮬레이션(`N_PATHS`개)으로 $P(T\le1000)$을 재귀식 값과 비교하시오. 편향 걷기는 왜 1차원인데도 일시적인가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$2n$걸음에 원점이려면 오른쪽 $n$번·왼쪽 $n$번이므로 $p_{2n}(0,0)=\binom{2n}n(pq)^n$. $G(s)=\sum_n\binom{2n}n(pqs^2)^n=(1-4pqs^2)^{-1/2}$ (E2의 이항급수에 $s^2\to4pqs^2$). $4pq=1-(p-q)^2$이므로 $G(1)=((p-q)^2)^{-1/2}=1/|p-q|<\infty$ ⇒ **일시**, $\rho=1-1/G(1)=1-|p-q|=0.9$.

왜 일시인가: 대수의 법칙으로 $S_n/n\to p-q=0.1>0$이니 걷기는 오른쪽으로 **떠나 버린다**. 대칭 걷기의 $p_{2n}\asymp n^{-1/2}$가 $(4pq)^n\cdot n^{-1/2}$로 **지수적**으로 작아지므로 급수가 수렴한다. 같은 이유로 꼬리가 지수적으로 가벼워 $P(T\le1000)=0.899987$은 이미 $\rho=0.9$에 붙어 있다 — 대칭 걷기의 $0.9748$ vs $1$과 대조적이다. 재귀성은 차원만이 아니라 **대칭성**의 문제이기도 하다.

코드 (SE $\approx0.0021$ at $n=20000$):
```python
p_r, q_r = 0.55, 0.45
pb = np.exp(gammaln(2 * nn + 1) - 2 * gammaln(nn + 1) + nn * np.log(p_r * q_r))   # C(2n,n)(pq)^n
fb = first_return_pmf(pb)
rho_biased = 1 - abs(p_r - q_r)                                  # 0.9
p_ret_biased_exact = float(fb.sum())                             # 0.899987
steps_b = np.where(rng.random((N_PATHS, N_STEPS)) < p_r, 1, -1)
S_b = steps_b.cumsum(1)
hit = S_b == 0
T_b = np.where(hit.any(1), hit.argmax(1) + 1, np.inf)
est_b = mc_proportion(np.isfinite(T_b))
print(f"rho = {rho_biased}, P(T<=1000) exact {p_ret_biased_exact:.6f}, MC {est_b}")
assert_close(est_b, p_ret_biased_exact, k=4, name="p_return_by_1000_biased")
```

</details>

## 7. 정리

1. 재귀 $\iff\sum_np^n(0,0)=\infty\iff$ 기대 방문 횟수 무한 (정리 1, 유한성 불필요); 일시이면 $\rho=1-1/G$.
2. $p_{2n}(0,0)\asymp n^{-d/2}$ (스털링): $d=1$ $(\pi n)^{-1/2}$, $d=2$ $(\pi n)^{-1}$, $d=3$ $0.2333\,n^{-3/2}$ ⇒ **Pólya**: 1·2차원 재귀, 3차원 이상 일시.
3. 3차원 귀환확률 $\rho_3=1-1/1.516386=0.3405$; 1000걸음 안 귀환 $0.9748/0.6782/0.3315$ ($d=1,2,3$) — 시뮬레이션과 4 SE 안에서 일치. 유한 지평의 비율은 $\rho$가 아니다.
4. $d=1$: $P(T>2n)=P(S_{2n}=0)\sim(\pi n)^{-1/2}$, $f_{2n}\sim\frac1{2\sqrt\pi}n^{-3/2}$, $ET=\infty$ — **영재귀**(정상분포 없음).
5. 편향 걷기는 1차원에서도 일시($\rho=1-|p-q|$): 재귀성은 대칭성과 차원 모두의 문제.

**trap 카드** — Q: 1차원 대칭 랜덤 워크는 재귀적이다. 그러면 평균 귀환시간 $E[T]$는 유한한가? 그리고 정상분포는 있는가?
→ A: 둘 다 아니다. $P(T=2n)\sim n^{-3/2}/(2\sqrt\pi)$이라 $\sum_n2n\,f_{2n}=\infty$ — 영재귀. $\pi(x)=1/E_xT_x=0$이 되어 정규화할 수 없으니 정상분포가 없다. 01d의 존재 정리에 '유한'이 붙은 이유.

**다음 노트북**: 01r — 모듈 01의 닫힌 책 복습: 정의·정리·숫자를 기억으로 재구성하고 세 가지 재유도(파산, 정상분포, 소멸확률)를 코드로 채점한다.

**log/progress.md 한 줄 템플릿**: `- [ ] 01x 완료 (YYYY-MM-DD) — 예측 맞힌 항목: _/7, 헷갈린 것: ____, 다음: 01r`